#  Airbnb Market Analysis

##  Objective: Analyze collected Airbnb listing data to provide insights on private rooms. Combining data from multiple types: .csv, .tvs, .xlsx

# Question 1: What are the dates of the earliest and most recent reviews?

In [83]:
# Import necessary packages
import pandas as pd

In [84]:
# Load and explore all files
price_df = pd.read_csv("../data/airbnb_price.csv")
print(price_df.head(), "\nColumns:Rows", price_df.shape)

room_df= pd.read_excel("../data/airbnb_room_type.xlsx")
print(room_df.head(), "\nColumns:Rows", room_df.shape)

review_df = pd.read_csv("../data/airbnb_last_review.tsv", sep="\t")
print(review_df.head(), "\nColumns:Row", review_df.shape)

   listing_id        price                nbhood_full
0        2595  225 dollars         Manhattan, Midtown
1        3831   89 dollars     Brooklyn, Clinton Hill
2        5099  200 dollars     Manhattan, Murray Hill
3        5178   79 dollars  Manhattan, Hell's Kitchen
4        5238  150 dollars       Manhattan, Chinatown 
Columns:Rows (25209, 3)
   listing_id                                description        room_type
0        2595                      Skylit Midtown Castle  Entire home/apt
1        3831            Cozy Entire Floor of Brownstone  Entire home/apt
2        5099  Large Cozy 1 BR Apartment In Midtown East  Entire home/apt
3        5178            Large Furnished Room Near B'way     private room
4        5238         Cute & Cozy Lower East Side 1 bdrm  Entire home/apt 
Columns:Rows (25209, 3)
   listing_id    host_name   last_review
0        2595     Jennifer   May 21 2019
1        3831  LisaRoxanne  July 05 2019
2        5099        Chris  June 22 2019
3        5178     

In [ ]:
# Cleaning data: standardizing capitalization, remove dollars and covert date type
room_df["room_type"] = room_df["room_type"].str.lower()
price_df["price"] = price_df["price"].astype(str)
price_df["price"] = price_df["price"].str.replace(" dollars","")
price_df["price"] = price_df["price"].astype(float)
review_df["last_review"] = pd.to_datetime(review_df["last_review"])

print(room_df["room_type"].head(3),
    "\n",
    review_df["last_review"].head(3), 
    "\n",
    price_df["price"].head(3))

0    entire home/apt
1    entire home/apt
2    entire home/apt
Name: room_type, dtype: object 
 0   2019-05-21
1   2019-07-05
2   2019-06-22
Name: last_review, dtype: datetime64[ns] 
 0    225.0
1     89.0
2    200.0
Name: price, dtype: float64


In [184]:
# Merge the three DataFrames into a single DataFrame  
merged_df = pd.merge(price_df,room_df, on='listing_id')
merged_df = pd.merge(merged_df,review_df, on='listing_id')
print(merged_df.head(), merged_df.shape)

   listing_id  price                nbhood_full  \
0        2595  225.0         Manhattan, Midtown   
1        3831   89.0     Brooklyn, Clinton Hill   
2        5099  200.0     Manhattan, Murray Hill   
3        5178   79.0  Manhattan, Hell's Kitchen   
4        5238  150.0       Manhattan, Chinatown   

                                 description        room_type    host_name  \
0                      Skylit Midtown Castle  entire home/apt     Jennifer   
1            Cozy Entire Floor of Brownstone  entire home/apt  LisaRoxanne   
2  Large Cozy 1 BR Apartment In Midtown East  entire home/apt        Chris   
3            Large Furnished Room Near B'way     private room     Shunichi   
4         Cute & Cozy Lower East Side 1 bdrm  entire home/apt          Ben   

  last_review  
0  2019-05-21  
1  2019-07-05  
2  2019-06-22  
3  2019-06-24  
4  2019-06-09   (25209, 7)


In [188]:
# Determine the earliest and most recent review dates
first_reviewed = merged_df["last_review"].min()
last_reviewed = merged_df["last_review"].max()
print("First review date:", first_reviewed.date())
print("Last review date:", last_reviewed.date())

First review date: 2019-01-01
Last review date: 2019-07-09


In [191]:
#Finding how many listings are private rooms
nb_private_rooms = (merged_df["room_type"] == "private room").sum()
print(nb_private_rooms)

11356


In [193]:
#Finding the average price of listings
avg_price = merged_df["price"].mean().round(2) # type: ignore
print("The average price:", avg_price) # type: ignore

The average price: 141.78


In [194]:
#Finding the average price of private rooms
avg_private_rooms_price = merged_df.loc[merged_df["room_type"] == "private room","price"].mean().round(2)
print("The average private room price:", avg_private_rooms_price)

#Query the table with list_id and price of private rooms
private_rooms_price_df = pd.merge(price_df,room_df, on ='listing_id', how='left')
private_room_table = private_rooms_price_df.loc[private_rooms_price_df["room_type"] == "private room"][["listing_id", "price"]]
print(private_room_table)


The average private room price: 81.64
       listing_id  price
3            5178   79.0
6            5441   85.0
7            5803   89.0
8            6021   85.0
11           7322  140.0
...           ...    ...
25201    36390226   45.0
25204    36425863  129.0
25205    36427429   45.0
25206    36438336  235.0
25208    36455809   30.0

[11356 rows x 2 columns]


In [195]:
#Create a Data Frame 
review_dates = pd.DataFrame({"first_reviewed": [first_reviewed],
                          "last_reviewd": [last_reviewed],
                          "nb_private_rooms": [nb_private_rooms],
                          "avg_price": [avg_price]}, index=[0])
print(review_dates)

  first_reviewed last_reviewd  nb_private_rooms  avg_price
0     2019-01-01   2019-07-09             11356     141.78
